# WALHUS, AMEMIYA, Swamy-Arora and nested error components

Random-effects panel regressions differ in how they estimate the variances of the error components before GLS. Julia packages offer Swamy-Arora for the one-way model at most; Wallace and Hussain (WALHUS), Amemiya (AMEMIYA), the two-way Swamy-Arora, and the estimators of the unbalanced nested model (states within regions) are not in any of them. All of them follow one rule: take residuals, form quadratic sums of squares, and equate each to its exact expectation. That rule is written out once here, the estimators are its choices of residuals, and they are tested against Baltagi's tables for the Grunfeld, gasoline and public capital data: Tables 2.1, 2.5, 2.6, 3.1, 3.6, 3.7 and 9.4 of *Econometric Analysis of Panel Data* (3rd edition, 2005).

The data is `grunfeld`, `gasoline` and `produc`, from PollisDatasets. Baltagi uses 10 of Grunfeld's 11 firms, without American Steel; Baltagi and Griffin's 18 OECD countries over 1960-78; and Munnell's 48 states over 1970-86 in 9 regions.

In [ ]:
Pollis.packages("PollisDatasets", "MixedModels", "Plots")

## Set-up

`replicate` prints each value in full, rounded to the published precision, beside the published value, and how many decimals they share.

In [ ]:
using PollisDatasets, MixedModels, Plots, LinearAlgebra, Printf, Statistics

"""
	replicate(title, rows)

Print each Julia value in full, rounded to the published precision and beside the published value, with the number of decimals they share.
Each row is `(label, value, published)`, with `published` written as in the source: its decimals give the published precision.
"""
function replicate(title, rows)
	@printf("\n%s\n%-26s%16s%14s%14s\n", title, "", "Julia", "rounded", "published")
	for (label, value, published) in rows
		p = parse(Float64, replace(published, "," => "", "−" => "-"))
		d = occursin('.', published) ? length(split(published, '.')[end]) : 0  # decimals as published
		k = findlast(j -> round(value, digits=j) == round(p, digits=j), 0:d)  # position in 0:d
		@printf("%-26s%16.8f%14s%14s  %s\n", label, value, Printf.format(Printf.Format("%.$(d)f"), value), published, k === nothing ? "DIFFERENT" : "same to $(k - 1) decimals")
	end
end

grunfeld = dataset("grunfeld");
firms = grunfeld.firm .!= "American Steel";  # Baltagi's 10 firms
invest, value, capital = grunfeld.invest[firms], grunfeld.value[firms], grunfeld.capital[firms];  # Grunfeld's I, F and K
gasoline = dataset("gasoline");  # 18 countries x 19 years
munnell = dataset("produc");  # 48 states x 17 years
lny = log.(munnell.gsp);  # log gross state product

## The method

### Error components and their projections

The disturbance is a sum of components: an individual effect, perhaps a time effect or a nested effect, and the remainder. Each component has an incidence matrix, its dummies, and the model splits the data into orthogonal parts with projections: within individuals, between individuals and, for two-way models, between periods; for nested models, within states, between states of a region, and between regions.

In [ ]:
"""The dummies of a grouping: one column per group."""
dummies(g) = Float64.(g .== permutedims(unique(g)))

"""The projection onto the columns of `W`; `pinv` allows designs with redundant columns."""
projection(W) = W * pinv(W)

"""
	projections(groups, structure)

The orthogonal projections that split the data, the within projection first: `:oneway` (individuals), `:twoway` (individuals, periods)
or `:nested` (regions, then states within them). Each is orthogonal to the overall mean.
"""
function projections(groups, structure)
	n = length(groups[1]); J = fill(1 / n, n, n)
	P = [projection(dummies(g)) for g in groups]
	structure == :oneway && return [I - P[1], P[1] - J]
	structure == :twoway && return [I - P[1] - P[2] + J, P[1] - J, P[2] - J]
	structure == :nested && return [I - P[2], P[2] - P[1], P[1] - J]
	error("structure must be :oneway, :twoway or :nested")
end

### One rule: quadratic forms equated to their expectations

Let e = My be residuals, where M removes the regressors, so e = Mu whatever the coefficients. A sum of squares q = e'Ae then has the exact expectation E(q) = Σ_c σ_c² tr(M'AM Z_c Z_c'), Z_c the incidence matrix of component c. With as many sums of squares as components, `anova` solves E(q) = q for the variances. This is how Baltagi defines the unbalanced estimators (Section 9.2.1, equations 9.16 to 9.21), and in balanced panels it gives the familiar formulas.

In [ ]:
"""
	anova(y, forms, incidence)

Variance components by equating each sum of squares `e'Ae`, with `e = M*y`, to its exact expectation. `forms` are the pairs `(A, M)`;
`incidence` are the incidence matrices of the components, the remainder (the identity) last.
"""
function anova(y, forms, incidence)
	q = [(e = M * y; transpose(e) * A * e) for (A, M) in forms]
	E = [(B = transpose(M) * A * M; tr(transpose(Z) * B * Z)) for (A, M) in forms, Z in incidence]
	E \ q
end

"""Residuals of the regression of `A*y` on `A*X`, as a matrix that acts on `y`."""
residualsof(A, X) = (AX = A * X; A - AX * pinv(AX))

### The estimators: which residuals

- **WALHUS** (Wallace and Hussain 1969): OLS residuals in every sum of squares.
- **AMEMIYA** (Amemiya 1971; Wansbeek and Kapteyn in the unbalanced case, WK): the residuals of the Within regression, y − α̃ − Xβ̃ with α̃ = ȳ − x̄β̃.
- **SWAR** (Swamy and Arora 1972, SA): each sum of squares from its own regression, Within for the remainder, Between individuals, Between periods or regions for the others.
- **HFB** (Henderson's method III, Fuller and Battese 1974): the fitting constants, reductions in the sum of squares as the nested dummies are added to the regression. Written here for the nested model, where Table 9.4 tests it.
- **MIVQUE** (Rao 1971) with a prior guess of the variances: quadratic forms weighted by the GLS residual maker under that guess.

A negative variance is set to zero, as in Tables 3.1 and 3.7. For WALHUS and AMEMIYA in balanced one-way and two-way models, `divisors = :trace` divides each sum of squares by the trace of its projection instead, as if the residuals were the true disturbances: equations (2.21), (2.22) and (3.17). That is what R's plm computes; Baltagi's tables use the exact expectations.

In [ ]:
"""
	variances(y, X, groups; structure=:oneway, method=:swar, prior=nothing)

The variances of the error components, in the order of `groups` and then the remainder, by `method` `:walhus`, `:amemiya`, `:swar`,
`:hfb` (nested only) or `:mivque` (with `prior`, a guess of the same variances). Negative estimates are set to zero. `divisors = :trace`
(WALHUS and AMEMIYA, one-way and two-way) divides each sum of squares by the trace of its projection, as R's plm does.
"""
function variances(y, X, groups; structure=:oneway, method=:swar, prior=nothing, divisors=:exact)
	n = length(y); Z = [ones(n) X]; J = fill(1 / n, n, n)
	A = projections(groups, structure)
	incidence = [[dummies(g) for g in groups]; [Matrix(1.0I, n, n)]]
	if method == :walhus
		Mols = I - projection(Z)
		forms = [(Ak, Mols) for Ak in A]
	elseif method == :amemiya
		Mwithin = (I - J) * (I - X * pinv(A[1] * X) * A[1])  # y - alpha - X b, alpha = ybar - xbar b
		forms = [(Ak, Mwithin) for Ak in A]
	elseif method == :swar
		forms = [(Ak, residualsof(Ak, X)) for Ak in A]
	elseif method == :hfb
		structure == :nested || error("HFB is written for the nested model")
		H = [projection([Z dummies(groups[2])]), projection([Z dummies(groups[1])]), projection(Z)]
		forms = [(I - H[1], I), (H[1] - H[2], I), (H[2] - H[3], I)]
	elseif method == :mivque
		V = sum(prior[k] * incidence[k] * transpose(incidence[k]) for k in eachindex(incidence))
		Vi = inv(V); R = Vi - Vi * Z * ((transpose(Z) * Vi * Z) \ (transpose(Z) * Vi))
		forms = [(R * Zk * transpose(Zk) * R, I) for Zk in incidence]
		E = [tr(R * Zi * transpose(Zi) * R * Zj * transpose(Zj)) for Zi in incidence, Zj in incidence]
		return max.(E \ [transpose(y) * A * y for (A, _) in forms], 0)
	else
		error("unknown method $method")
	end
	if divisors == :trace
		method in (:walhus, :amemiya) && structure != :nested || error("trace divisors are for WALHUS and AMEMIYA, one-way and two-way")
		structure == :oneway && (forms[2] = (forms[2][1] + J, forms[2][2]))  # P, whose trace is N (2.21)
		q = [(e = M * y; transpose(e) * Ak * e) for (Ak, M) in forms]
		E = [tr(transpose(Zc) * Ak * Zc) for (Ak, _) in forms, Zc in incidence]  # expectations of the true disturbances
		return max.(E \ q, 0)
	end
	max.(anova(y, forms, incidence), 0)
end

### Feasible GLS

With the variances, Ω = Σ_c σ_c² Z_c Z_c', and GLS is OLS on the data premultiplied by the inverse Cholesky factor of Ω. The standard errors come from (Z'Ω⁻¹Z)⁻¹. With `scale = true` they are multiplied by the residual variance of the transformed regression, as EViews does: Baltagi's two-way tables are computed that way.

In [ ]:
"""
	errorcomponents(y, X, groups; structure=:oneway, method=:swar, prior=nothing, divisors=:exact, scale=false)

Feasible GLS of `y` on an intercept and `X` with error components: the coefficients, their standard errors and the variances
(see `variances`). `scale = true` multiplies the covariance matrix by the residual variance of the transformed regression.
"""
function errorcomponents(y, X, groups; structure=:oneway, method=:swar, prior=nothing, divisors=:exact, scale=false)
	n = length(y); Z = [ones(n) X]
	s = variances(y, X, groups; structure, method, prior, divisors)
	incidence = [[dummies(g) for g in groups]; [Matrix(1.0I, n, n)]]
	C = cholesky(Symmetric(sum(s[k] * incidence[k] * transpose(incidence[k]) for k in eachindex(incidence))))
	Zt, yt = C.L \ Z, C.L \ y
	V = inv(transpose(Zt) * Zt); b = V * (transpose(Zt) * yt)
	scale && (V *= sum(abs2, yt - Zt * b) / (n - size(Z, 2)))
	(coef = b, se = sqrt.(diag(V)), variances = s)
end

## Tests

`compare` lays out one estimator's row of a published table: the slopes and their standard errors, then the components. The tables print standard deviations; Table 9.4 prints variances and the intercept.

In [ ]:
"""
	compare(title, fit, published; names, components, intercept=false, variance=false, rho=false)

Replicate one row of a published table: `published` is `(coefficients, standard errors, components)` as printed, `names` labels the
coefficients and `components` the error components, in the table's order (indices into `fit.variances`). Standard deviations unless
`variance`; `rho` adds sigma_mu^2 / (sigma_mu^2 + sigma_nu^2) for one-way tables.
"""
function compare(title, fit, published; names, components, intercept=false, variance=false, rho=false)
	b, se, c = published
	k = intercept ? (1:length(fit.coef)) : (2:length(fit.coef))
	rows = Any[]
	for (j, i) in enumerate(k)
		push!(rows, (names[j], fit.coef[i], b[j]))
		isempty(se) || se[j] == "" || push!(rows, ("  se", fit.se[i], se[j]))
	end
	v = fit.variances
	rho && push!(rows, ("rho", v[1] / (v[1] + v[end]), c[1]))
	for (j, (label, i)) in enumerate(components)
		push!(rows, (label, variance ? v[i] : sqrt(v[i]), c[j + rho]))
	end
	replicate(title, rows)
end

### Grunfeld, one-way (Table 2.1, p. 21)

I = α + β₁F + β₂K + u, with firm effects.

In [ ]:
X = [value capital]; groups = [grunfeld.firm[firms]]
table21 = Dict(:walhus => (["0.110", "0.308"], ["0.011", "0.017"], ["0.73", "87.36", "53.75"]),
	:amemiya => (["0.110", "0.308"], ["0.010", "0.017"], ["0.71", "83.52", "52.77"]),
	:swar => (["0.110", "0.308"], ["0.010", "0.017"], ["0.72", "84.20", "52.77"]))
for method in (:walhus, :amemiya, :swar)
	compare("$(uppercase(string(method))), Table 2.1", errorcomponents(invest, X, groups; method), table21[method];
		names=["F", "K"], components=[("sigma_mu", 1), ("sigma_nu", 2)], rho=true)
end

### Gasoline, one-way (Table 2.5, p. 24)

log(Gas/Car) = α + β₁ log(Y/N) + β₂ log(PMG/PGDP) + β₃ log(Car/N) + u, with country effects: income, real gasoline price and cars per capita.

In [ ]:
X = [gasoline.lincomep gasoline.lrpmg gasoline.lcarpcap]; groups = [gasoline.country]
names = ["log Y/N", "log PMG/PGDP", "log Car/N"]
table25 = Dict(:walhus => (["0.545", "-0.447", "-0.605"], ["0.066", "0.046", "0.029"], ["0.75", "0.197", "0.113"]),
	:amemiya => (["0.602", "-0.366", "-0.621"], ["0.066", "0.042", "0.027"], ["0.93", "0.344", "0.092"]),
	:swar => (["0.555", "-0.402", "-0.607"], ["0.059", "0.042", "0.026"], ["0.82", "0.196", "0.092"]))
for method in (:walhus, :amemiya, :swar)
	compare("$(uppercase(string(method))), Table 2.5", errorcomponents(gasoline.lgaspcar, X, groups; method), table25[method];
		names, components=[("sigma_mu", 1), ("sigma_nu", 2)], rho=true)
end

### Public capital, one-way (Table 2.6, p. 25)

log Q = α + β₁ log KG + β₂ log KP + β₃ log L + β₄ Unemp + u, with state effects: KG public and KP private capital.

In [ ]:
X = [log.(munnell.pcap) log.(munnell.pc) log.(munnell.emp) munnell.unemp]; groups = [munnell.state]
names = ["log KG", "log KP", "log L", "Unemp"]
table26 = Dict(:walhus => (["0.006", "0.311", "0.728", "-0.006"], ["0.024", "0.020", "0.025", "0.001"], ["0.82", "0.082", "0.039"]),
	:amemiya => (["0.002", "0.309", "0.733", "-0.006"], ["0.024", "0.020", "0.025", "0.001"], ["0.84", "0.088", "0.038"]),
	:swar => (["0.004", "0.311", "0.730", "-0.006"], ["0.023", "0.020", "0.025", "0.001"], ["0.82", "0.083", "0.038"]))
for method in (:walhus, :amemiya, :swar)
	compare("$(uppercase(string(method))), Table 2.6", errorcomponents(lny, X, groups; method), table26[method];
		names, components=[("sigma_mu", 1), ("sigma_nu", 2)], rho=true)
end

### Grunfeld, two-way (Table 3.1, p. 43)

Firm and year effects. WALHUS and SWAR estimate a negative year variance, set to zero.

In [ ]:
X = [value capital]; groups = [grunfeld.firm[firms], grunfeld.year[firms]]
table31 = Dict(:walhus => (["0.110", "0.308"], ["0.010", "0.017"], ["87.31", "0", "55.33"]),
	:amemiya => (["0.111", "0.324"], ["0.011", "0.019"], ["89.26", "15.78", "51.72"]),
	:swar => (["0.110", "0.308"], ["0.011", "0.017"], ["84.23", "0", "51.72"]))
for method in (:walhus, :amemiya, :swar)
	compare("$(uppercase(string(method))), Table 3.1", errorcomponents(invest, X, groups; structure=:twoway, method, scale=true), table31[method];
		names=["F", "K"], components=[("sigma_mu", 1), ("sigma_lambda", 2), ("sigma_nu", 3)])
end

### Gasoline, two-way (Table 3.6, p. 46)

Country and year effects. WALHUS and SWAR estimate a negative year variance, set to zero.

In [ ]:
X = [gasoline.lincomep gasoline.lrpmg gasoline.lcarpcap]; groups = [gasoline.country, gasoline.year]
names = ["log Y/N", "log PMG/PGDP", "log Car/N"]
table36 = Dict(:walhus => (["0.545", "-0.450", "-0.605"], ["0.056", "0.039", "0.025"], ["0.197", "0", "0.115"]),
	:amemiya => (["0.170", "-0.233", "-0.602"], ["0.080", "0.041", "0.026"], ["0.423", "0.131", "0.081"]),
	:swar => (["0.565", "-0.405", "-0.609"], ["0.061", "0.040", "0.026"], ["0.196", "0", "0.081"]))
for method in (:walhus, :amemiya, :swar)
	compare("$(uppercase(string(method))), Table 3.6", errorcomponents(gasoline.lgaspcar, X, groups; structure=:twoway, method, scale=true), table36[method];
		names, components=[("sigma_mu", 1), ("sigma_lambda", 2), ("sigma_nu", 3)])
end

### Public capital, two-way (Table 3.7, p. 47)

In [ ]:
X = [log.(munnell.pcap) log.(munnell.pc) log.(munnell.emp) munnell.unemp]; groups = [munnell.state, munnell.year]
names = ["log KG", "log KP", "log L", "Unemp"]
table37 = Dict(:walhus => (["0.026", "0.258", "0.742", "-0.005"], ["0.023", "0.021", "0.024", "0.001"], ["0.082", "0.016", "0.036"]),
	:amemiya => (["0.002", "0.217", "0.770", "-0.004"], ["0.025", "0.024", "0.026", "0.001"], ["0.154", "0.026", "0.034"]),
	:swar => (["0.018", "0.266", "0.745", "-0.005"], ["0.023", "0.021", "0.024", "0.001"], ["0.083", "0.010", "0.034"]))
for method in (:walhus, :amemiya, :swar)
	compare("$(uppercase(string(method))), Table 3.7", errorcomponents(lny, X, groups; structure=:twoway, method, scale=true), table37[method];
		names, components=[("sigma_mu", 1), ("sigma_lambda", 2), ("sigma_nu", 3)])
end

### Public capital, nested (Table 9.4, p. 182)

log Q = α + β₁ log K + β₂ log L + β₃ log KH + β₄ log KW + β₅ log KO + β₆ Unemp + u, with region effects μ, state effects ν nested in them and the remainder ε; public capital split into highways (KH), water (KW) and other (KO). The book does not give the priors of MV1 to MV3; MV1 is tested with (0, 0, 1), MV2 with (1, 1, 1) (see What differs).

In [ ]:
X = [log.(munnell.pc) log.(munnell.emp) log.(munnell.hwy) log.(munnell.water) log.(munnell.util) munnell.unemp]
groups = [munnell.region, munnell.state]  # regions, then states
names = ["Intercept", "log K", "log L", "log KH", "log KW", "log KO", "Unemp"]
table94 = [("WH", (method=:walhus,), ["2.082", "0.273", "0.742", "0.075", "0.076", "-0.095", "-0.006"], ["0.152", "0.021", "0.026", "0.023", "0.014", "0.017", "0.001"], ["0.0014", "0.0027", "0.0045"]),
	("WK", (method=:amemiya,), ["2.131", "0.264", "0.758", "0.072", "0.076", "-0.102", "-0.006"], ["0.160", "0.022", "0.027", "0.024", "0.014", "0.017", "0.001"], ["0.0014", "0.0022", "0.0069"]),
	("SA", (method=:swar,), ["2.089", "0.274", "0.740", "0.073", "0.076", "-0.094", "-0.006"], ["0.144", "0.020", "0.025", "0.022", "0.014", "0.017", "0.001"], ["0.0014", "0.0015", "0.0043"]),
	("HFB", (method=:hfb,), ["2.084", "0.272", "0.743", "0.075", "0.076", "-0.096", "-0.006"], ["0.150", "0.021", "0.026", "0.022", "0.014", "0.017", "0.001"], ["0.0014", "0.0029", "0.0044"]),
	("MV1", (method=:mivque, prior=[0, 0, 1]), ["2.083", "0.272", "0.742", "0.075", "0.076", "-0.095", "-0.006"], ["0.152", "0.021", "0.026", "0.023", "0.014", "0.017", "0.001"], ["0.0014", "0.0027", "0.0046"]),
	("MV2", (method=:mivque, prior=[1, 1, 1]), ["2.114", "0.269", "0.750", "0.072", "0.076", "-0.098", "-0.006"], ["0.154", "0.021", "0.026", "0.023", "0.014", "0.017", "0.001"], ["0.0014", "0.0017", "0.0056"])]
nested = Dict{String, Any}()
for (label, options, b, se, c) in table94
	nested[label] = errorcomponents(lny, X, groups; structure=:nested, options...)
	compare("$label, Table 9.4", nested[label], (b, se, c); names, intercept=true, variance=true,
		components=[("sigma_epsilon^2", 3), ("sigma_mu^2", 1), ("sigma_nu^2", 2)])
end

The MLE and REML columns of the same table are not new: MixedModels.jl fits them. They check the data and the model.

In [ ]:
data = (lny = lny, k = X[:, 1], l = X[:, 2], kh = X[:, 3], kw = X[:, 4], ko = X[:, 5], unemp = X[:, 6], region = string.(munnell.region), state = munnell.state)
formula = @formula(lny ~ 1 + k + l + kh + kw + ko + unemp + (1 | region) + (1 | state))
for (label, reml, b, se, c) in (("MLE", false, ["2.129", "0.267", "0.754", "0.071", "0.076", "-0.100", "-0.006"], ["0.154", "0.021", "0.026", "0.023", "0.014", "0.017", "0.001"], ["0.0013", "0.0015", "0.0063"]),
		("REML", true, ["2.127", "0.266", "0.756", "0.072", "0.076", "-0.101", "-0.006"], ["0.157", "0.022", "0.026", "0.023", "0.014", "0.017", "0.001"], ["0.0014", "0.0019", "0.0064"]))
	m = fit(MixedModel, formula, data; REML=reml, progress=false)
	v = [VarCorr(m).σρ.region.σ[1]^2, VarCorr(m).σρ.state.σ[1]^2, m.σ^2]
	compare("$label (MixedModels.jl), Table 9.4", (coef = coef(m), se = stderror(m), variances = v), (b, se, c); names, intercept=true, variance=true,
		components=[("sigma_epsilon^2", 3), ("sigma_mu^2", 1), ("sigma_nu^2", 2)])
end

### Cross-check with R's plm

plm fits WALHUS, AMEMIYA and SWAR, one-way, two-way and nested. This cell runs the same 15 models in R, if `Rscript` and plm are installed, and prints the largest relative difference from Julia in the coefficients, standard errors and variances. plm scales its standard errors by the residual variance of the transformed regression, so Julia uses `scale = true` here. For WALHUS and AMEMIYA in the one-way and two-way models, Julia uses `divisors = :trace`, plm's definition.

Differences of about 1e-7 are R printing 7 significant digits. The two-way WALHUS and AMEMIYA rows differ more, because there plm divides the between-individuals and between-periods sums of squares by N and T rather than by their traces N − 1 and T − 1 (equation 3.17). Rescaling them reproduces plm's variances, for example 5685.23 for the Grunfeld firms with WALHUS and 243.78 for the years with AMEMIYA. Neither convention reproduces Baltagi's Table 3.1 or 3.7, which the exact expectations do.

In [ ]:
function plmcheck()
	Sys.which("Rscript") === nothing && return println("Rscript not found: install R and plm to run the cross-check")
	dir = mktempdir()
	open(joinpath(dir, "grunfeld.csv"), "w") do io
		println(io, "firm,year,invest,value,capital")
		for i in findall(firms)
			println(io, join((repr(grunfeld.firm[i]), grunfeld.year[i], grunfeld.invest[i], grunfeld.value[i], grunfeld.capital[i]), ","))
		end
	end
	open(joinpath(dir, "produc.csv"), "w") do io
		println(io, "state,year,region,pcap,hwy,water,util,pc,gsp,emp,unemp")
		for i in eachindex(munnell.state)
			println(io, join((repr(munnell.state[i]), munnell.year[i], munnell.region[i], munnell.pcap[i], munnell.hwy[i], munnell.water[i], munnell.util[i], munnell.pc[i], munnell.gsp[i], munnell.emp[i], munnell.unemp[i]), ","))
		end
	end
	write(joinpath(dir, "check.R"), """
	suppressMessages(library(plm))
	g <- read.csv("grunfeld.csv"); p <- read.csv("produc.csv")
	show <- function(label, m) {
	  s <- summary(m)\$coefficients; v <- ercomp(m)\$sigma2
	  cat(label, s[, 1], s[, 2], v[c(2:length(v), 1)], "\n")  # coefficients, standard errors, variances (idiosyncratic last)
	}
	for (meth in c("walhus", "amemiya", "swar")) {
	  for (eff in c("individual", "twoways")) {
	    show(paste("Grunfeld", eff, meth), plm(invest ~ value + capital, g, index = c("firm", "year"), model = "random", effect = eff, random.method = meth))
	    show(paste("Produc", eff, meth), plm(log(gsp) ~ log(pcap) + log(pc) + log(emp) + unemp, p, index = c("state", "year"), model = "random", effect = eff, random.method = meth))
	  }
	  show(paste("Produc nested", meth), plm(log(gsp) ~ log(pc) + log(emp) + log(hwy) + log(water) + log(util) + unemp, p, index = c("state", "year", "region"), model = "random", effect = "nested", random.method = meth))
	}
	""")
	lines = try
		readlines(Cmd(`Rscript check.R`; dir))
	catch
		return println("R could not run plm: install it with install.packages(\"plm\")")
	end
	Xg = [value capital]; Xp = [log.(munnell.pcap) log.(munnell.pc) log.(munnell.emp) munnell.unemp]
	Xn = [log.(munnell.pc) log.(munnell.emp) log.(munnell.hwy) log.(munnell.water) log.(munnell.util) munnell.unemp]
	println("Largest relative difference from plm")
	@printf("%-32s%14s%14s%14s\n", "", "coefficients", "std. errors", "variances")
	for line in lines
		words = split(line); data, effect, method = words[1], words[2], Symbol(words[3]); r = parse.(Float64, words[4:end])
		structure = effect == "individual" ? :oneway : effect == "twoways" ? :twoway : :nested
		divisors = method != :swar && structure != :nested ? :trace : :exact
		y, X, groups = data == "Grunfeld" ? (invest, Xg, structure == :oneway ? [grunfeld.firm[firms]] : [grunfeld.firm[firms], grunfeld.year[firms]]) :
			structure == :nested ? (lny, Xn, [munnell.region, munnell.state]) :
			(lny, Xp, structure == :oneway ? [munnell.state] : [munnell.state, munnell.year])
		f = errorcomponents(y, X, groups; structure, method, divisors, scale=true)
		v = structure == :nested ? f.variances[[2, 1, 3]] : f.variances  # plm: state, region, remainder
		k = length(f.coef); largest(a, b) = maximum(abs.(a - b) ./ abs.(b .+ (b .== 0)))  # relative, absolute where plm gives 0
		@printf("%-32s%14.1e%14.1e%14.1e\n", "$data $effect $method", largest(f.coef, r[1:k]), largest(f.se, r[k+1:2k]), largest(v, r[2k+1:end]))
	end
end
plmcheck()

## Example

Does public capital in highways raise private output? In the nested model, the six estimators split the error variance between regions and states in very different ways, σμ² from .0015 to .0029 and σν² from .0043 to .0069, yet the highways coefficient and its 95% interval hardly move.

In [ ]:
labels = ["WH", "WK", "SA", "HFB", "MV1", "MV2"]
kh = [nested[l].coef[4] for l in labels]; halfwidth = [1.96 * nested[l].se[4] for l in labels]
scatter(labels, kh, yerror=halfwidth, label=nothing, ylabel="log KH coefficient", title="Highways, 95% intervals", size=(560, 330))
hline!([0], color=:gray, linestyle=:dash, label=nothing)

## What differs

Tables 2.1, 2.6, 3.1, 3.6 and 3.7, and the WALHUS and AMEMIYA rows of Table 2.5, agree with the published values to every printed digit: coefficients, standard errors, ρ and the standard deviations of the components. Two details of Baltagi's computations had to be matched. His standard errors in the two-way tables (3.1, 3.6, 3.7) are scaled by the residual variance of the transformed regression, while those of the one-way tables and Table 9.4 are not: with the other convention, about a third of the standard errors differ in the last digit. And the sums of squares are equated to their exact expectations, the Chapter 9 form, not divided by the traces of equations (2.21) and (2.22): for AMEMIYA's remainder this gives SWAR's divisor N(T − 1) − K rather than N(T − 1), which is why the two share σν = 52.77 in Table 2.1. It also reproduces the EViews output for WALHUS in Table 2.2: σμ = 87.358027 and σν = 53.745175 here, against 87.35803 and 53.74518.

Table 9.4: WH, WK, SA and MV1 and MV2 agree to every printed digit, and so do MLE and REML from MixedModels.jl. The rest:

- **HFB, in the last digit**: intercept 2.088 against 2.084, log K .271 against .272, log L .744 against .743, se of the intercept .151 against .150, σμ² .0027 against .0029, σν² .0046 against .0044. The fitting-constants reductions here follow Section 9.2.1; Baltagi, Song and Jung (2001) define the nested version, which the book does not reproduce, and it may order the reductions differently.
- **MIVQUE priors**: the book does not say which prior variances MV1 to MV3 use. Priors (0, 0, 1) and (1, 1, 1) reproduce MV1 and MV2 exactly; no prior tried (the WH, WK, SA, HFB, ML, MV1 and MV2 estimates, and iteration to convergence) reproduces MV3, whose closest, the WH estimates, gives σμ² .0018 against .0017, so MV3 is not tested.

**R's plm** agrees with Julia to R's printed precision (7 significant digits) for SWAR in all three structures, for all three nested estimators, and for one-way WALHUS and AMEMIYA, which plm computes with the trace divisors of equations (2.21) and (2.22). So plm does not reproduce Baltagi's Table 2.1 for WALHUS and AMEMIYA, nor Table 2.6 (σμ = .078 against .082 for WALHUS on the public capital data, and 75.43 against 87.36 on Grunfeld). For its two-way WALHUS and AMEMIYA, see the cross-check above.

**Table 2.5, SWAR row**: the variances agree (ρ .82, σμ .196, σν .092), and so do the income and cars coefficients, but the price coefficient is −.420 here against −.402, and the standard errors are .057, .039 and .025 against .059, .042 and .026. Scaled by the residual variance, as in the two-way tables, they become .059, .040 and .026. R's plm gives the same row as Julia with scaled standard errors: −.4204, and .059, .040 and .0255. So −.402 looks like transposed digits of −.420; the standard error .042 is not explained.

## References

- Amemiya, T. (1971). The estimation of the variances in a variance-components model. *International Economic Review* 12(1), 1-13.
- Baltagi, B. H. (2005). *Econometric Analysis of Panel Data*, 3rd edition. Wiley. Tables 2.1, 2.5, 2.6, 3.1, 3.6, 3.7 and 9.4.
- Baltagi, B. H. and Griffin, J. M. (1983). Gasoline demand in the OECD: An application of pooling and testing procedures. *European Economic Review* 22(2), 117-137.
- Baltagi, B. H., Song, S. H. and Jung, B. C. (2001). The unbalanced nested error component regression model. *Journal of Econometrics* 101(2), 357-381. doi:10.1016/S0304-4076(00)00089-0
- Fuller, W. A. and Battese, G. E. (1974). Estimation of linear models with crossed-error structure. *Journal of Econometrics* 2(1), 67-78.
- Grunfeld, Y. (1958). *The Determinants of Corporate Investment*. PhD thesis, University of Chicago.
- Munnell, A. H. (1990). How does public infrastructure affect regional economic performance? *New England Economic Review*, September/October, 11-32.
- Swamy, P. A. V. B. and Arora, S. S. (1972). The exact finite sample properties of the estimators of coefficients in the error components regression models. *Econometrica* 40(2), 261-275.
- Wallace, T. D. and Hussain, A. (1969). The use of error components models in combining cross section with time series data. *Econometrica* 37(1), 55-72.